# MHQA — Retrieval-Augmented Reader (Qwen2.5-7B-Instruct QLoRA) : SFT + GRPO on COMB

Subsets `['Lug_Uga','Eng_Eth','Eng_Ken','Eng_Uga','Swa_Ken']`. Metric COMB = 0.5*R1 + 0.5*RL.

**Why this breaks past 0.766:** retrieval only gives recall@1≈0.7 but recall@50≈0.9 — the answer is *in* the
top-K, the bi-encoder just can't rank it #1, and a cross-encoder couldn't either. So instead of *selecting*,
a **trained reader reads Q + the top-K candidate answers** and (a) copies the right one verbatim (ROUGE→1),
or (b) composes when none fits (the only way to move Eng_Ket/Eng_Eth past the copy wall).

**Training:** SFT (learn copy-or-compose from references) -> **GRPO with reward = COMB** (optimize the exact,
non-differentiable metric; the answer is in-context so high reward is reachable). Per-subset dev-gate vs the
afrie5 copy baseline -> never regresses below ~0.766.

**Dependencies (attach as a Kaggle wheels dataset, internet off):** transformers, accelerate, peft,
bitsandbytes, trl, datasets, sentence-transformers, rouge-score. If any are missing the corresponding stage
is skipped gracefully and you still get the afrie5-copy submission.

In [1]:
import os, sys, subprocess
WHEELS_DIR = '/kaggle/input/datasets/haniagamal/teleco-wheels/wheels'   # must contain the LLM stack wheels
def _pip(pkgs):
    if not WHEELS_DIR: return
    try: subprocess.run([sys.executable,'-m','pip','install','-q','--no-index',f'--find-links={WHEELS_DIR}',*pkgs],check=True); print('installed',pkgs)
    except Exception as e: print('offline pip note', pkgs, str(e)[:80])
_pip(['rouge-score','peft','bitsandbytes','trl','datasets','accelerate'])
for k in ['HF_HUB_OFFLINE','TRANSFORMERS_OFFLINE','HF_DATASETS_OFFLINE']: os.environ[k]='1'
os.environ.setdefault('TOKENIZERS_PARALLELISM','false')
import re, gc, random, unicodedata, warnings
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')
SEED=42; random.seed(SEED); np.random.seed(SEED)

HAVE_ST=HAVE_HF=HAVE_PEFT=HAVE_BNB=HAVE_TRL=HAVE_DS=False; DEVICE='cpu'
try:
    import torch; DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
    from sentence_transformers import SentenceTransformer, InputExample, losses
    from torch.utils.data import DataLoader; HAVE_ST=True
except Exception as e: print('[info] sentence-transformers off:',type(e).__name__)
try:
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig; HAVE_HF=True
except Exception as e: print('[info] transformers off:',type(e).__name__)
try:
    import peft; from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training; HAVE_PEFT=True
except Exception as e: print('[info] peft off:',type(e).__name__)
try:
    import bitsandbytes; HAVE_BNB=True
except Exception as e: print('[info] bitsandbytes off:',type(e).__name__)
try:
    import trl; from trl import SFTTrainer, SFTConfig, GRPOTrainer, GRPOConfig; HAVE_TRL=True; print('trl',trl.__version__)
except Exception as e: print('[info] trl off:',type(e).__name__)
try:
    from datasets import Dataset as HFDataset; HAVE_DS=True
except Exception as e: print('[info] datasets off:',type(e).__name__)
print('device=%s | ST=%s HF=%s peft=%s bnb=%s trl=%s ds=%s'%(DEVICE,HAVE_ST,HAVE_HF,HAVE_PEFT,HAVE_BNB,HAVE_TRL,HAVE_DS))
import inspect, dataclasses
def _cfg(cls, **kw):                         # build a config passing ONLY kwargs this TRL version accepts
    valid=set()
    try:
        if dataclasses.is_dataclass(cls): valid|={f.name for f in dataclasses.fields(cls)}
    except Exception: pass
    try: valid|=set(inspect.signature(cls.__init__).parameters)
    except Exception: pass
    drop=[k for k in kw if valid and k not in valid]
    if drop: print('  [cfg] %s dropping unsupported: %s'%(getattr(cls,'__name__','cfg'),drop))
    return cls(**({k:v for k,v in kw.items() if (not valid) or k in valid}))

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cudf-cu12 26.2.1 requires numba-cuda[cu12]<0.23.0,>=0.22.2, but you hav

installed ['rouge-score', 'peft', 'bitsandbytes', 'trl', 'datasets', 'accelerate']
trl 1.2.0
device=cuda | ST=True HF=True peft=True bnb=True trl=True ds=True


In [2]:
# =============================== CONFIG ===============================
DRY_RUN = False             # True = tiny end-to-end validation in minutes (proves the pipeline). Set False for the real run.
# --- retriever that produces the candidate set (the proven afrie5 bi-encoder) ---
RET = {'path':'/kaggle/input/datasets/haniagamal/mhqa-models-data/models/afrie5','q_prefix':'','d_prefix':''}
SELECT_SUBSETS = ['Lug_Uga','Eng_Eth'] #,'Eng_Ken','Eng_Uga','Swa_Ken']
CAND_TOPK   = 10           # candidate answers shown to the reader
FT_EPOCHS=2; FT_BATCH=64; FT_LR=2e-5; FT_MAX_SEQ=256; PAIRS_PER_CLUSTER=30; MAX_TRAIN_PAIRS=25000
HARD_NEG_TOPK=8; HARD_NEGS_PER_PAIR=4

# --- reader : Qwen2.5-7B-Instruct + QLoRA (4-bit) ---  (best-supported model for TRL GRPO)
READER_PATH = '/kaggle/input/models/qwen-lm/qwen2.5/transformers/3b-instruct/1'   # << set to YOUR attached path (3B-Instruct also fine + faster)
USE_4BIT    = False
LORA_R=16; LORA_ALPHA=32; LORA_DROPOUT=0.05
LORA_TARGETS=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj']   # Qwen2.5 attn+MLP
MAX_PROMPT_LEN=1536; MAX_NEW=256

# --- SFT ---
SFT_EPOCHS=1; SFT_LR=2e-4; SFT_BSZ=1; SFT_ACCUM=8

# --- GRPO (reward = COMB) ---
USE_GRPO=True
GRPO_NUMGEN=4; GRPO_LR=1e-6; GRPO_BETA=0.04; GRPO_BSZ=4; GRPO_ACCUM=4; GRPO_STEPS=300; GRPO_MAXNEW=256
GRPO_TRAIN_Q=3000          # pool queries used for GRPO (per_device batch is forced to a multiple of num_generations)

# --- routing / gating ---
GEN_LENGTH_CALIBRATE=True
DEV_EVAL_N=300             # dev rows generated for the after-SFT / after-GRPO COMBINED checks (7B gen is slow -> sample)
DEV_FRAC=0.15
WORK_DIR=Path('/kaggle/working') if os.path.isdir('/kaggle/working') else Path('.')
try: WORK_DIR.mkdir(parents=True,exist_ok=True)
except Exception: WORK_DIR=Path('.')
OUTPUT_SUBMISSION=WORK_DIR/'submission_rag_reader.csv'
if DRY_RUN:                # shrink everything so the WHOLE pipeline (SFT+GRPO+gate+submit) runs in minutes
    SFT_EPOCHS=1; GRPO_STEPS=6; GRPO_TRAIN_Q=GRPO_NUMGEN*8; CAND_TOPK=6
    print('*** DRY_RUN ON: tiny validation pass; set DRY_RUN=False for the full run ***')
print('reader=%s | cand_topk=%d | grpo=%s | dry_run=%s'%(READER_PATH.split('/')[-1],CAND_TOPK,USE_GRPO,DRY_RUN))

reader=1 | cand_topk=10 | grpo=True | dry_run=False


In [3]:
# data
DATA_DIR=Path('/kaggle/input/datasets/haniagamal/repo-mhqa/multilingual-health-qa/data')
ID_COL,QUESTION_COL,ANSWER_COL,LANG_COL='ID','input','output','subset'
def clean_text(x): return '' if pd.isna(x) else str(x).strip()
train=pd.read_csv(DATA_DIR/'Train.csv'); val=pd.read_csv(DATA_DIR/'Val.csv'); test=pd.read_csv(DATA_DIR/'Test.csv')
for df in (train,val):
    df[QUESTION_COL]=df[QUESTION_COL].map(clean_text); df[ANSWER_COL]=df[ANSWER_COL].map(clean_text)
test[QUESTION_COL]=test[QUESTION_COL].map(clean_text)
full=pd.concat([train,val],ignore_index=True)
full=full[(full[QUESTION_COL]!='')&(full[ANSWER_COL]!='')]
full=full.drop_duplicates(subset=[QUESTION_COL,ANSWER_COL,LANG_COL]).reset_index(drop=True)
full=full[full[LANG_COL].isin(SELECT_SUBSETS)].reset_index(drop=True)
_n0=len(test); test=test[test[LANG_COL].isin(SELECT_SUBSETS)].reset_index(drop=True)
SUBS=[s for s in SELECT_SUBSETS if s in set(full[LANG_COL])]
try:
    from sklearn.model_selection import train_test_split
    _pi,_di=train_test_split(np.arange(len(full)),test_size=DEV_FRAC,random_state=SEED,stratify=full[LANG_COL].values)
except Exception:
    rng=np.random.RandomState(SEED); pm=rng.permutation(len(full)); cut=int(len(full)*DEV_FRAC); _di,_pi=pm[:cut],pm[cut:]
pool=full.iloc[_pi].reset_index(drop=True); dev=full.iloc[_di].reset_index(drop=True)
if DRY_RUN:
    pool=pool.groupby(LANG_COL,group_keys=False).head(60).reset_index(drop=True)
    dev =dev.groupby(LANG_COL,group_keys=False).head(20).reset_index(drop=True)
    test=test.groupby(LANG_COL,group_keys=False).head(8).reset_index(drop=True)
print('full=%d pool=%d dev=%d test=%d/%d'%(len(full),len(pool),len(dev),len(test),_n0))

full=8708 pool=7401 dev=1307 test=434/2618


In [4]:
# metric COMB + reward helpers
def _utok(s):
    s=unicodedata.normalize('NFKC',str(s)).lower()
    return [x for x in re.sub(r'[^\w]+',' ',s,flags=re.UNICODE).split() if x]
def _lcs(a,b):
    if not a or not b: return 0
    p=[0]*(len(b)+1)
    for i in range(1,len(a)+1):
        c=[0]*(len(b)+1); ai=a[i-1]
        for j in range(1,len(b)+1): c[j]=p[j-1]+1 if ai==b[j-1] else max(p[j],c[j-1])
        p=c
    return p[-1]
def _f1(o,n,r): return 0.0 if (o==0 or n==0 or r==0) else 2*(o/n)*(o/r)/((o/n)+(o/r))
def pair_scores(ref,pred):
    rt,pt=_utok(ref),_utok(pred)
    return _f1(sum((Counter(rt)&Counter(pt)).values()),len(pt),len(rt)), _f1(_lcs(rt,pt),len(pt),len(rt))
def _comb1(ref,pred):
    r1,rl=pair_scores(ref,pred); return 0.5*r1+0.5*rl
def combined(preds,refs):
    s=[pair_scores(r,p) for p,r in zip(preds,refs)]
    return {'COMB':float(np.mean([0.5*a+0.5*b for a,b in s]))}
def _norm(s): return ' '.join(_utok(s))
REF_LEN={}
for s in SUBS:
    ol=[len(_utok(a)) for a in full[full[LANG_COL]==s][ANSWER_COL].values]; REF_LEN[s]=int(np.median(ol)) if ol else 60
def _cal(text,s):
    if not GEN_LENGTH_CALIBRATE: return text
    toks=str(text).split(); cap=int(REF_LEN.get(s,60)*1.6)+8
    return ' '.join(toks[:cap]) if len(toks)>cap else text
print('metric + reward ready | REF_LEN', REF_LEN)

metric + reward ready | REF_LEN {'Lug_Uga': 78, 'Eng_Eth': 24}


In [5]:
# retriever (afrie5) -> candidate answers
class DenseRetriever:
    def __init__(self,m): self.m=m
    def _enc(self,texts,p): return np.asarray(self.m.encode([p+str(t) for t in texts],batch_size=128,normalize_embeddings=True,show_progress_bar=False),dtype='float32')
    def fit(self,bank_q,d): self.D=self._enc(bank_q,d)
    def sims(self,q,p): return self._enc(q,p)@self.D.T
class TfidfRetriever:
    def fit(self,bank_q,d=''):
        from sklearn.feature_extraction.text import TfidfVectorizer
        self.v=TfidfVectorizer(analyzer='char_wb',ngram_range=(3,5),min_df=2); self.D=self.v.fit_transform([str(x) for x in bank_q])
    def sims(self,q,p=''): return np.asarray((self.v.transform([str(x) for x in q])@self.D.T).todense())

def finetune_retriever(df_bank):
    if not HAVE_ST:
        r=TfidfRetriever(); r.fit(df_bank[QUESTION_COL].values); print('  [retriever] TFIDF fallback'); return r
    base=SentenceTransformer(RET['path'],device=DEVICE); base.max_seq_length=FT_MAX_SEQ
    miner=DenseRetriever(base); miner.fit(df_bank[QUESTION_COL].values,RET['d_prefix'])
    qs=df_bank[QUESTION_COL].values; ans=df_bank[ANSWER_COL].values
    a2=defaultdict(list)
    for i,a in enumerate(ans): a2[a].append(i)
    rng=np.random.RandomState(SEED); raw=[]
    for rows in [v for v in a2.values() if len(v)>=2]:
        cand=[(rows[i],rows[j]) for i in range(len(rows)) for j in range(i+1,len(rows))]
        if len(cand)>PAIRS_PER_CLUSTER: cand=[cand[k] for k in rng.choice(len(cand),PAIRS_PER_CLUSTER,replace=False)]
        raw+=cand
    if len(raw)>MAX_TRAIN_PAIRS: raw=[raw[k] for k in rng.choice(len(raw),MAX_TRAIN_PAIRS,replace=False)]
    S=miner.sims([qs[a] for a in sorted({a for a,_ in raw})],RET['q_prefix']) if raw else None
    anchors=sorted({a for a,_ in raw}); hn={}
    if raw:
        order=np.argsort(-S,axis=1)[:, :HARD_NEG_TOPK+HARD_NEGS_PER_PAIR+4]
        for ai,a in enumerate(anchors):
            own=set(a2[ans[a]]); negs=[j for j in order[ai] if j not in own and j!=a][:HARD_NEGS_PER_PAIR]
            if negs: hn[a]=negs
    ex=[]
    for a,b in raw:
        for x,y in ((a,b),(b,a)):
            t=[RET['q_prefix']+str(qs[x]),RET['d_prefix']+str(qs[y])]
            for n in hn.get(x,[]): t.append(RET['d_prefix']+str(qs[n]))
            ex.append(InputExample(texts=t))
    dl=DataLoader(ex,shuffle=True,batch_size=FT_BATCH,drop_last=True)
    loss=losses.MultipleNegativesRankingLoss(base)
    base.fit(train_objectives=[(dl,loss)],epochs=FT_EPOCHS,warmup_steps=int(0.1*len(dl)*FT_EPOCHS),optimizer_params={'lr':FT_LR},show_progress_bar=True)
    print('  [retriever] fine-tuned on %d examples'%len(ex)); return DenseRetriever(base)

def get_candidates(ret, bank_df, query_qs, K, drop_self=False):
    ret.fit(bank_df[QUESTION_COL].values,RET['d_prefix']); S=ret.sims(query_qs,RET['q_prefix'])
    ba=bank_df[ANSWER_COL].values; out=[]; kk=min(K+ (1 if drop_self else 0)+5, S.shape[1])
    idx=np.argpartition(-S,kk-1,axis=1)[:,:kk]
    for i in range(len(query_qs)):
        order=idx[i][np.argsort(-S[i,idx[i]])]; seen=set(); cands=[]; top1sim=float(S[i,order[0]])
        for j in order:
            a=str(ba[j]); n=_norm(a)
            if drop_self and n==_norm(query_qs[i]): pass
            if n in seen: continue
            seen.add(n); cands.append(a)
            if len(cands)>=K: break
        out.append((cands, top1sim))
    return out
print('retriever + candidate builder ready')

retriever + candidate builder ready


In [6]:
# prompt builder (chat) + copy baseline
SYS=('You are a multilingual health assistant. Answer the question in the SAME language. '
     'If one candidate answer below correctly and completely answers the question, reply with it EXACTLY, verbatim. '
     'Otherwise write a concise, accurate answer.')
def build_user(q, cands):
    lines='\n'.join('%d. %s'%(i+1,c) for i,c in enumerate(cands))
    return 'Question: %s\n\nCandidate answers:\n%s\n\nFinal answer:'%(q, lines)
def to_prompt(tok, user, add_gen=True):
    msgs=[{'role':'user','content':SYS+'\n\n'+user}]   # fold system into user (universal: safe for templates w/ or w/o a system role)
    try: return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=add_gen)
    except Exception: return SYS+'\n\n'+user+'\n'
def copy_baseline(ret, bank_df, qdf):
    cs=get_candidates(ret,bank_df,qdf[QUESTION_COL].values,1)
    return [c[0][0] if c[0] else '' for c in cs]
print('prompt builder ready')

prompt builder ready


In [7]:
# fine-tune retriever on POOL (candidates for SFT/GRPO/dev are built from pool; full for test)
RET_POOL=finetune_retriever(pool)
import gc; gc.collect()
if HAVE_ST and DEVICE=='cuda':
    import torch; torch.cuda.empty_cache()

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss


  [retriever] fine-tuned on 15734 examples


In [8]:
# load reader (Qwen2.5-7B-Instruct) in 4-bit + LoRA
READER=None; RTOK=None
if HAVE_HF and HAVE_PEFT:
    try:
        import torch
        bnb=None
        if USE_4BIT and HAVE_BNB:
            bnb=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type='nf4',
                                   bnb_4bit_compute_dtype=torch.bfloat16,bnb_4bit_use_double_quant=True)
        RTOK=AutoTokenizer.from_pretrained(READER_PATH)
        if RTOK.pad_token is None: RTOK.pad_token=RTOK.eos_token
        RTOK.padding_side='left'   # REQUIRED for correct batched generation with a decoder-only model
        READER=AutoModelForCausalLM.from_pretrained(READER_PATH,quantization_config=bnb,
               device_map='auto',torch_dtype=torch.bfloat16,trust_remote_code=True)
        READER=prepare_model_for_kbit_training(READER)
        lcfg=LoraConfig(r=LORA_R,lora_alpha=LORA_ALPHA,lora_dropout=LORA_DROPOUT,bias='none',
                        task_type='CAUSAL_LM',target_modules=LORA_TARGETS)
        READER=get_peft_model(READER,lcfg); READER.print_trainable_parameters()
        print('reader loaded:',READER_PATH)
    except Exception as e:
        print('[warn] reader load failed -> copy baseline only:',type(e).__name__,str(e)[:160]); READER=None
else:
    print('reader: transformers/peft unavailable -> copy baseline only')

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

trainable params: 29,933,568 || all params: 3,115,872,256 || trainable%: 0.9607
reader loaded: /kaggle/input/models/qwen-lm/qwen2.5/transformers/3b-instruct/1


In [9]:
# ===== reader helpers + SFT (copy-or-compose) + DEV COMBINED check AFTER SFT (before GRPO) =====
def make_examples(qdf, ret, bank_df, with_ref=True):
    cs=get_candidates(ret,bank_df,qdf[QUESTION_COL].values,CAND_TOPK, drop_self=True)
    rows=[]
    for i in range(len(qdf)):
        cands,sim=cs[i]; user=build_user(qdf[QUESTION_COL].values[i],cands)
        rec={'prompt':to_prompt(RTOK,user,add_gen=True),'subset':qdf[LANG_COL].values[i],'sim':sim}
        if with_ref:
            ref=str(qdf[ANSWER_COL].values[i]); rec['reference']=ref
            rec['completion']=' '+ref+(getattr(RTOK,'eos_token',None) or '')
        rows.append(rec)
    return rows

def reader_generate(rows, batch=8):
    if READER is None: return ['']*len(rows)
    import torch; READER.eval(); RTOK.padding_side='left'; out=[]   # left-pad for decoder-only generation
    for k in range(0,len(rows),batch):
        chunk=rows[k:k+batch]; prompts=[r['prompt'] for r in chunk]
        enc=RTOK(prompts,return_tensors='pt',padding=True,truncation=True,max_length=MAX_PROMPT_LEN).to(READER.device)
        with torch.no_grad():
            g=READER.generate(**enc,max_new_tokens=MAX_NEW,do_sample=False,num_beams=1,pad_token_id=RTOK.pad_token_id)
        for i in range(len(chunk)):
            txt=RTOK.decode(g[i][enc['input_ids'].shape[1]:],skip_special_tokens=True).strip()
            out.append(_cal(txt, chunk[i]['subset']))
    return out

def _dev_sample():
    if len(dev)<=DEV_EVAL_N: return dev
    per=max(1,DEV_EVAL_N//max(1,len(SUBS)))
    return dev.groupby(LANG_COL,group_keys=False).head(per).reset_index(drop=True)
def eval_reader_dev(tag):
    if READER is None: return -1.0
    try:
        d=_dev_sample(); preds=reader_generate(make_examples(d,RET_POOL,pool,with_ref=False))
        c=combined(preds, list(d[ANSWER_COL].values))['COMB']
        print('  >>> DEV COMBINED (%s) = %.4f   [0.5*R1+0.5*RL] over %d dev rows'%(tag,c,len(d))); return c
    except Exception as e:
        print('  [dev-eval warn] %s: %s'%(type(e).__name__,str(e)[:120])); return -1.0
def _snap():
    try:
        from peft import get_peft_model_state_dict
        return {k:v.detach().cpu().clone() for k,v in get_peft_model_state_dict(READER).items()}
    except Exception as e: print('  [snap warn]',type(e).__name__); return None
def _restore(state):
    try:
        from peft import set_peft_model_state_dict; set_peft_model_state_dict(READER,state); return True
    except Exception as e: print('  [restore warn]',type(e).__name__); return False

COMB_SFT=-1.0; COMB_GRPO=-1.0; SFT_STATE=None
if READER is not None and HAVE_TRL and HAVE_DS:
    try:
        sft_rows=make_examples(pool, RET_POOL, pool, with_ref=True)
        ds=HFDataset.from_list([{'prompt':r['prompt'],'completion':r['completion']} for r in sft_rows])  # prompt masked
        base_kw=dict(output_dir=str(WORK_DIR/'sft'),num_train_epochs=SFT_EPOCHS,per_device_train_batch_size=SFT_BSZ,
              gradient_accumulation_steps=SFT_ACCUM,learning_rate=SFT_LR,logging_steps=50,save_strategy='no',
              bf16=(DEVICE=='cuda'),report_to=[],gradient_checkpointing=True,
              gradient_checkpointing_kwargs={'use_reentrant':False})
        scfg=_cfg(SFTConfig, max_length=MAX_PROMPT_LEN+MAX_NEW, max_seq_length=MAX_PROMPT_LEN+MAX_NEW, **base_kw)
        tr=None
        for tk in ('processing_class','tokenizer'):
            try: tr=SFTTrainer(model=READER,args=scfg,train_dataset=ds,**{tk:RTOK}); break
            except TypeError: continue
        if tr is None: tr=SFTTrainer(model=READER,args=scfg,train_dataset=ds)
        tr.train(); print('SFT done on %d examples (prompt-completion)'%len(ds))
        COMB_SFT=eval_reader_dev('after SFT, before GRPO'); SFT_STATE=_snap()
    except Exception as e:
        print('[warn] SFT failed:',type(e).__name__,str(e)[:200])
else:
    print('SFT skipped (reader/trl/datasets unavailable)')

  [cfg] SFTConfig dropping unsupported: ['max_seq_length']


Adding EOS to train dataset:   0%|          | 0/7401 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/7401 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
50,0.024658
100,0.005591
150,0.006063
200,0.006443
250,0.006225
300,0.006136
350,0.005168
400,0.004958
450,0.005612
500,0.006440


SFT done on 7401 examples (prompt-completion)


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


  >>> DEV COMBINED (after SFT, before GRPO) = 0.5088   [0.5*R1+0.5*RL] over 300 dev rows


In [10]:
# ===== GRPO : optimize reward = COMB(generation, reference) directly =====
def comb_reward(prompts=None, completions=None, reference=None, subset=None, **kw):
    comp=completions or []
    outs=[(c if isinstance(c,str) else (c[-1]['content'] if c else '')) for c in comp]
    refs=reference if reference is not None else ['']*len(outs)
    subs=subset if subset is not None else ['']*len(outs)
    return [float(_comb1(refs[i], _cal(outs[i], subs[i]))) for i in range(len(outs))]

if READER is not None and HAVE_TRL and HAVE_DS and USE_GRPO:
    try:
        rng=np.random.RandomState(SEED); sel=rng.choice(len(pool),int(min(GRPO_TRAIN_Q,len(pool))),replace=False)
        grpo_rows=make_examples(pool.iloc[sel].reset_index(drop=True), RET_POOL, pool, with_ref=True)
        gds=HFDataset.from_list([{'prompt':r['prompt'],'reference':r['reference'],'subset':r['subset']} for r in grpo_rows])
        g=int(GRPO_NUMGEN); bsz=g*max(1,int(GRPO_BSZ)//g)     # per-device batch MUST be a multiple of num_generations
        gcfg=_cfg(GRPOConfig, output_dir=str(WORK_DIR/'grpo'),learning_rate=GRPO_LR,beta=GRPO_BETA,
              per_device_train_batch_size=bsz,gradient_accumulation_steps=GRPO_ACCUM,
              num_generations=g,max_prompt_length=MAX_PROMPT_LEN,max_completion_length=GRPO_MAXNEW,
              max_steps=GRPO_STEPS,logging_steps=10,save_strategy='no',bf16=(DEVICE=='cuda'),report_to=[],
              gradient_checkpointing=True,gradient_checkpointing_kwargs={'use_reentrant':False})
        gtr=None
        for tk in ('processing_class','tokenizer'):
            try: gtr=GRPOTrainer(model=READER,reward_funcs=[comb_reward],args=gcfg,train_dataset=gds,**{tk:RTOK}); break
            except TypeError: continue
        if gtr is None: gtr=GRPOTrainer(model=READER,reward_funcs=[comb_reward],args=gcfg,train_dataset=gds)
        gtr.train(); print('GRPO done')
        COMB_GRPO=eval_reader_dev('after GRPO')      # validate AFTER GRPO and keep the better of SFT/GRPO
        if SFT_STATE is not None and COMB_GRPO < COMB_SFT:
            if _restore(SFT_STATE): print('  >>> GRPO (%.4f) did NOT beat SFT (%.4f) -> REVERTED to SFT adapter'%(COMB_GRPO,COMB_SFT))
        else:
            print('  >>> keeping post-GRPO model (GRPO=%.4f vs SFT=%.4f)'%(COMB_GRPO,COMB_SFT))
    except Exception as e:
        print('[warn] GRPO failed (keeping SFT reader):',type(e).__name__,str(e)[:200])
else:
    print('GRPO skipped')

  [cfg] GRPOConfig dropping unsupported: ['max_prompt_length']


Step,Training Loss
10,-0.000690
20,0.001115
30,-0.025565
40,0.004529
50,-0.006762
60,-0.000406
70,-0.013171
80,-0.011889
90,-0.005425
100,-0.004643


GRPO done
  >>> DEV COMBINED (after GRPO) = 0.5083   [0.5*R1+0.5*RL] over 300 dev rows
  >>> GRPO (0.5083) did NOT beat SFT (0.5088) -> REVERTED to SFT adapter


In [11]:
# ===== per-subset dev gate (reader vs afrie5 copy) on the kept model + headline DEV COMBINED =====
USE_READER={s:False for s in SUBS}
dev_copy=copy_baseline(RET_POOL, pool, dev)              # afrie5 copy for ALL dev rows in ONE batched pass
dev_refs=list(dev[ANSWER_COL].values)
if READER is not None:
    print('per-subset dev gate | COMB = 0.5*R1 + 0.5*RL  (afrie5 copy vs RAG reader):')
    dev_rows=make_examples(dev, RET_POOL, pool, with_ref=True)
    rd=reader_generate(dev_rows)
    routed=list(dev_copy)                                # start from copy, swap to reader where it wins
    for s in SUBS:
        idx=[i for i in range(len(dev)) if dev[LANG_COL].values[i]==s]
        if not idx: continue
        refs=[dev_refs[i] for i in idx]
        ccopy=combined([dev_copy[i] for i in idx],refs)['COMB']
        cread=combined([rd[i] for i in idx],refs)['COMB']
        USE_READER[s]=cread>ccopy
        if USE_READER[s]:
            for i in idx: routed[i]=rd[i]
        print('  %-9s copy=%.4f  reader=%.4f  -> %s'%(s,ccopy,cread,'READER' if USE_READER[s] else 'copy'))
    ov=combined(routed, dev_refs)['COMB']
    print('  '+'-'*54)
    print('  >>> DEV COMBINED (routed, weighted overall) = %.4f   [0.5*R1 + 0.5*RL]'%ov)
else:
    ov=combined(dev_copy, dev_refs)['COMB']
    print('gate skipped (no reader) -> afrie5 copy baseline')
    print('  >>> DEV COMBINED (afrie5 copy, weighted overall) = %.4f   [0.5*R1 + 0.5*RL]'%ov)

per-subset dev gate | COMB = 0.5*R1 + 0.5*RL  (afrie5 copy vs RAG reader):
  Lug_Uga   copy=0.6826  reader=0.3368  -> copy
  Eng_Eth   copy=0.6744  reader=0.6696  -> copy
  ------------------------------------------------------
  >>> DEV COMBINED (routed, weighted overall) = 0.6784   [0.5*R1 + 0.5*RL]


In [12]:
# ===== FINAL : retriever on FULL -> route per subset (reader where it won, else afrie5 copy) -> submission =====
RET_FULL=finetune_retriever(full)
ans={}
for s in SUBS:
    q=test[test[LANG_COL]==s].reset_index(drop=True)
    if len(q)==0: continue
    bank=full[full[LANG_COL]==s].reset_index(drop=True)
    if USE_READER.get(s) and READER is not None:
        rows=make_examples(q, RET_FULL, bank, with_ref=False); preds=reader_generate(rows)
    else:
        preds=copy_baseline(RET_FULL, bank, q)
    for i,rid in enumerate(q[ID_COL].values): ans[rid]=preds[i]
    print('  %-9s n=%-5d -> %s'%(s,len(q),'reader' if USE_READER.get(s) else 'copy'))
out=[str(ans.get(rid,'')).strip() or (str(qq).strip()[:300] or 'N/A') for rid,qq in zip(test[ID_COL].values,test[QUESTION_COL].values)]
sub=pd.DataFrame({'ID':test[ID_COL].values,'TargetRLF1':out,'TargetR1F1':out,'TargetLLM':out})[['ID','TargetRLF1','TargetR1F1','TargetLLM']]
assert len(sub)==len(test) and (sub['TargetRLF1'].str.len()>0).all()
sub.to_csv(OUTPUT_SUBMISSION,index=False,encoding='utf-8')
print('saved %s shape=%s'%(OUTPUT_SUBMISSION,sub.shape))

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss
500,0.509767


  [retriever] fine-tuned on 20438 examples
  Lug_Uga   n=374   -> copy
  Eng_Eth   n=60    -> copy
saved /kaggle/working/submission_rag_reader.csv shape=(434, 4)
